In [1]:
from sklearn.cluster import KMeans
from graphgallery.datasets import NPZDataset
from graphgallery import functional as gf
import graphgallery as gg
import numpy as np
import torch
import pandas as pd
import numba
import random
from numba import njit
import tensorflow as tf
import os
import networkx as nx
import inspect
from gpu_mem_track import MemTracker
import pickle
import scipy.sparse as sp
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" 

'neighbor_sampler_cpu' is not enabled, maybe you should re-install graphgallery again.


In [2]:
# chameleon
# squirrel
# texas
dn = 'film'
data = NPZDataset(dn,
                      root="~/GraphData/datasets/",
                      verbose=False,
                      transform='standardize')

graph = data.graph
adj_matrix = graph.adj_matrix
indices = adj_matrix.indices
indptr = adj_matrix.indptr
node_attr = graph.node_attr
node_attr2 = gf.get('normalize_attr')(node_attr)
labels = graph.node_label
n_classes = len(set(labels))
splits = data.split_nodes(random_state=1)
seed = 2022
random.seed(seed)
targets = random.sample(list(splits.test_nodes), 50)
n_nodes = list(range(graph.adj_matrix.shape[0]))
print(n_classes)
nums = len(labels)

5


In [3]:
adj_matrix

<7600x7600 sparse matrix of type '<class 'numpy.float32'>'
	with 53318 stored elements in Compressed Sparse Row format>

In [5]:
node_attr.shape

(7600, 932)

In [ ]:
z shape:  (2485, 7)
[[-0.24526456 -0.7323458  -0.1569242  ... -1.3074628   1.0784162
   0.61009276]
 [-0.68851054 -0.896708    3.3563938  ... -1.1619074  -1.5896504
  -0.3557513 ]
 [ 3.1174378   0.20585741 -0.08731209 ... -0.5013163  -0.89025533
  -1.4878687 ]
 ...
 [-0.72213346 -2.2686875   3.5634325  ... -1.2906703  -1.6547651
   0.02976378]
 [-0.58033836 -2.0447667   4.378234   ... -1.3514106  -1.9048405
   0.01267643]
 [-0.46613765 -1.6028737   2.579746   ... -0.6723361  -1.1728153
  -0.18462655]]
<class 'numpy.ndarray'>
float32

In [ ]:
adj_matrix

In [64]:
model = gg.gallery.nodeclas.GCN(seed=seed).setup_graph(graph).build()
model.fit(splits.train_nodes, splits.val_nodes, verbose=1, epochs=200)
results = model.evaluate(splits.test_nodes, verbose=1)

Training...
200/200 [==============================] - Total: 35.45s - 177ms/step- loss: 0.298 - accuracy: 0.995 - val_loss: 0.358 - val_accuracy: 0.995
Testing...
1/1 [====================] - Total: 46.73ms - 46ms/step- loss: 0.652 - accuracy: 0.995


In [7]:
candidates = np.array(splits.test_nodes)
softmax_logits = model.predict(np.arange(adj_matrix.shape[0]), transform="softmax")
sur_labels = np.array([lo.argmax() for lo in softmax_logits])
ground_truths = graph.node_label
predict_suc_idx = sur_labels[candidates] == ground_truths[candidates]
test_nodes = candidates[predict_suc_idx]
target_nums = 100
# assert cmd.target_nums >= len(test_nodes), "the number of target nodes is larger than predicted suc nodes"
targets = random.sample(list(test_nodes), target_nums)

In [10]:
predict_suc_idx

array([ True,  True,  True, ...,  True,  True, False])

# deeprobust

In [19]:
from deeprobust.graph.defense import RGCN
from deeprobust.graph.data import Dataset
data = Dataset(root='C://Users/pc/tmp/', name='cora', setting='prognn')
adj, features, labels = data.adj, data.features, data.labels
idx_train, idx_val, idx_test = data.idx_train, data.idx_val, data.idx_test
model = RGCN(nnodes=adj.shape[0], nfeat=features.shape[1], nclass=labels.max()+1,
                    nhid=32, device="cpu")

model = model.to("cpu")

Loading cora dataset...
Selecting 1 largest connected components


In [20]:
model.fit(features, adj, labels, idx_train, idx_val, train_iters=10, verbose=True)

=== training rgcn model ===
Epoch 0, training loss: 8.43726634979248
=== picking the best model according to the performance on validation ===


In [21]:
model.output.max(1)[1].numpy()

array([6, 2, 0, ..., 2, 2, 2], dtype=int64)

# 选点统计

In [5]:
def load_json(filename):
    if 'npy' not in filename:
        filename += '.npy'
    return np.load(filename, allow_pickle=True).item()

def get_wrong_labels(logits, targets, labels):
    wrong_labels = []
    for target in targets:
        logit = logits[target]
        idx = list(set(range(logit.size)) - set([labels[target]]))
        wrong_label = idx[logit[idx].argmax()]
        wrong_labels.append(wrong_label)
    wrong_labels = np.array(wrong_labels)
    return wrong_labels

def get_statis(embed_types, js, wrong_labels, degs, labels, sur_labels, sur_pro_labels):
    statis = {}
    for et in embed_types:
        et_map = js[et]
        targets = list(js[et].keys())
        t2p = {}
        for i, target in enumerate(targets):
            perturbed_nodes = []
            for _, per in et_map[target].keys():
                perturbed_nodes.append(per)
            perturbed_nodes = np.array(perturbed_nodes)
            # wrong labels
            key = '_'.join([str(target), 'target_wrong_label'])
            t2p[key] = wrong_labels[i]
            
            # degree
            key = '_'.join([str(target), 'perturbed_nodes_degs'])
            t2p[key] = degs[perturbed_nodes]

            # ground truth
            key = '_'.join([str(target), 'perturbed_nodes_ground_truth'])
            t2p[key] = labels[perturbed_nodes]

            # sur labels
            key = '_'.join([str(target), 'perturbed_nodes_sur_labels'])
            t2p[key] = sur_labels[perturbed_nodes]

            # sur labels pro
            key = '_'.join([str(target), 'perturbed_nodes_sur_labels_pro'])
            t2p[key] = sur_pro_labels[perturbed_nodes]

            # wrong labels pro
            statis[et] = t2p
    return statis

def get_df(statis, embed_types):
    df_d = {}
    for et in embed_types:
        _set = statis[et]
        l = 0
        for arr in _set.values():
            if isinstance(arr, np.int32):
                continue
            l = max(l, len(arr))
        df = pd.DataFrame(columns=[i for i in range(l)])
        for key in _set.keys():
            if not isinstance(_set[key], np.int32):
                df.loc[key] = list(_set[key]) + [np.nan] * (l - len(_set[key]))
            else:
                df.loc[key] = [_set[key]] + [np.nan] * (l - 1)
        df_d[et] = df
    return df_d

def main(model_name, filename):
    js = load_json('result/test_edges/'+filename)
    embed_types = ['sga', 'MLP','GCN2','SGC2','FastGCN']
    model = gg.gallery.nodeclas.SGC(seed=1000).setup_graph(graph).build()
    model.model = model.model.load(model_name)
    degs = adj_matrix.toarray().sum(axis=0)
    targets = list(js['sga'].keys())
    logits = model.model.lin(model.cache.X).detach().cpu().numpy()
    sf_logits = gf.get('softmax')(logits)
    sur_labels = np.array([logit.argmax() for logit in sf_logits])
    sur_pro_labels = np.array([round(logit.max(), 3) for logit in sf_logits])
    wrong_labels = get_wrong_labels(logits, targets, graph.node_label)
    statis = get_statis(embed_types, js, wrong_labels, degs, labels, sur_labels, sur_pro_labels)
    df_d = get_df(statis, embed_types)
    return df_d
    # sur_pro_wrong_labels = [sf_logitswrong_labels[i]for i in range(len(targets))]

In [6]:
dn = 'chameleon'
data = NPZDataset(dn,
                      root="~/GraphData/datasets/",
                      verbose=False,
                      transform='standardize')

graph = data.graph
adj_matrix = graph.adj_matrix
indices = adj_matrix.indices
indptr = adj_matrix.indptr
node_attr = graph.node_attr
node_attr2 = gf.get('normalize_attr')(node_attr)
labels = graph.node_label
n_classes = len(set(labels))
splits = data.split_nodes(random_state=1)
seed = 2022
random.seed(seed)
targets = random.sample(list(splits.test_nodes), 50)
n_nodes = list(range(graph.adj_matrix.shape[0]))
print(n_classes)
nums = len(labels)

5


In [10]:
# 欧式
# df_d1 = main('cora_SGC_model', 'cora_2021_12_26_17_35_01_0.npy')
# 误分类
# df_d2 = main('cora_SGC_model', 'cora_2021_12_26_20_52_19_0.npy')

# 欧式
df_d1 = main('chameleon_SGC_model', 'chameleon_2021_12_26_21_09_37_0.npy')
# 误分类
df_d2 = main('chameleon_SGC_model', 'chameleon_2021_12_26_21_06_42_0.npy')

In [3]:
# df_d2['sga']

In [4]:
# df_d1['MLP']

In [5]:
# df_d2['MLP']

In [6]:
# df_d1['GCN2']